<a href="https://colab.research.google.com/github/AfnanAbdul/LLM-eval-framework-comparison/blob/main/notebooks/Sentiment_Analysis_Using_RoBERTA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sentiment Analysis Using RoBERTa

## Upload Dataset

In [ ]:
from google.colab import files
uploaded = files.upload()


Saving Tone Identification - Generated Dataset.csv to Tone Identification - Generated Dataset.csv


## Install Required Libraries

In [ ]:
!pip install transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 39.9 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

## Load and Prepare Data

In [ ]:
import pandas as pd

# Load the uploaded CSV file
tone_df = pd.read_csv("Tone Identification - Generated Dataset.csv")

# Ensure the necessary columns are present
tone_df = tone_df[['Text', 'Overall Tone']].dropna()

## Load RoBERTa Sentiment Pipeline

In [ ]:
from transformers import pipeline

# Load the RoBERTa sentiment analysis pipeline
sentiment_pipeline = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Device set to use cpu


## Apply RoBERTa to Data

In [ ]:
# Define a function to get sentiment predictions
def get_roberta_sentiment(text):
    result = sentiment_pipeline(text)[0]
    return result['label'], result['score']

# Apply the function to your DataFrame
tone_df[['RoBERTa_Predicted', 'RoBERTa_Confidence']] = tone_df['Text'].apply(get_roberta_sentiment).apply(pd.Series)


## Evaluate RoBERTa's Performance

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

# Map RoBERTa labels to match your ground truth labels if necessary
label_mapping = {
    'LABEL_0': 'negative',
    'LABEL_1': 'neutral',
    'LABEL_2': 'positive'
}

# Apply the mapping
tone_df['RoBERTa_Predicted'] = tone_df['RoBERTa_Predicted'].map(label_mapping)

#Apply cleaning
tone_df['Overall Tone'] = tone_df['Overall Tone'].str.strip().str.lower()
tone_df['RoBERTa_Predicted'] = tone_df['RoBERTa_Predicted'].str.strip().str.lower()

# Calculate accuracy
accuracy = accuracy_score(tone_df['Overall Tone'], tone_df['RoBERTa_Predicted'])
print(f"Accuracy: {accuracy:.2f}")

# Generate a classification report
print(classification_report(tone_df['Overall Tone'], tone_df['RoBERTa_Predicted']))


Accuracy: 0.78
              precision    recall  f1-score   support

    negative       0.88      0.90      0.89        41
     neutral       0.56      0.23      0.32        22
    positive       0.73      0.97      0.83        36

    accuracy                           0.78        99
   macro avg       0.72      0.70      0.68        99
weighted avg       0.75      0.78      0.74        99



In [ ]:
tone_df[["Text", "Overall Tone","RoBERTa_Predicted"]]

,Text,Overall Tone,RoBERTa_Predicted
0,I can't believe how amazing this concert is!,Positive,positive
1,"Ugh, the traffic today is unbearable.",Negative,negative
2,Could you be any more incompetent?,Negative,negative
3,Thank you so much for your assistance.,Positive,positive
4,Hearing about her loss breaks my heart.,Negative,negative
...,...,...,...
94,The weight of the situation was evident on eve...,Negative,neutral
95,"Every detail has been considered, and I couldn...",Positive,positive
96,Feel free to reach out if you have any other q...,Neutral,neutral
97,Why do I always have to be the one to fix thes...,Negative,negative


## Save files

In [ ]:
tone_df.to_csv("tone_df_with_roberta.csv", index=False)